# 二部グラフ GNN 提出（特徴量固定・モデル改良）

**方針**: 特徴量は現状の最高精度で固定し、**モデルだけ**を二部グラフ GNN（SAGEConv + LayerNorm）に差し替えてスコアを出す。

**内容**: train+test の (critic, movie) 接続のみでグラフを構築し、接続情報だけで GNN を学習。Fresh/Rotten を BCE で予測し、提出 CSV を保存する。

**参照**: `docs/16_NEXT_HEAVY_IMPROVEMENTS.md`（独自 GNN）

**依存**: `pip install torch torch_geometric`（未導入なら `pip install -r requirements.txt` を先に実行）

## 1. セットアップ

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

from lib.improvement_candidates import get_setup, run_gnn_bipartite_submission

ctx = get_setup(seed=42, output_dir="outputs", use_best_pipeline=True)
print(f"Train: {len(ctx.train):,}, Test: {len(ctx.test):,}")
print(f"提出先: {ctx.submissions_dir}")

## 2. GNN 学習・予測・提出 CSV 保存

In [ ]:
# 二部グラフ GNN（接続のみ、特徴量は使わない）
r = run_gnn_bipartite_submission(
    ctx,
    hidden_dim=64,
    num_layers=2,
    lr=1e-2,
    epochs=200,
    seed=42,
    out_name="submission_gnn_bipartite.csv",
)

if r.get("path"):
    p = r["path"]
    name = Path(p).name if isinstance(p, str) else p.name
    print(f"  [GNN 二部グラフ] → {name}  ({'OK' if r.get('ok') else r.get('message', '')})")
else:
    print(f"  [GNN 二部グラフ] スキップ: {r.get('message', '')}")

## 3. 提出ファイル確認

In [ ]:
path = ctx.submissions_dir / "submission_gnn_bipartite.csv"
print(f"  {'✓' if path.exists() else '✗'} {path.name}")